# Day 07 미니 프로젝트 · 나만의 에이전트 완성

오늘 배운 걸로 **작은 업무 비서 에이전트**를 직접 완성한다.
빈칸(TODO)을 채우면, 이 에이전트는 스스로 도구를 골라 쓰고 · 대화를 기억하고 · 에러를 만나면 고쳐 쓴다.

**채울 빈칸 (Part 2~5)** — 막히면 괄호 안의 실습 Lab을 다시 본다.
- Part 2 · 도구 변환 `to_openai_tools` (Lab 1)
- Part 3 · 에이전트 루프 `run_agent` — 판단→행동→관찰·종료조건·자기수정 (Lab 1·2·4)
- Part 4 · 메모리 `Agent` 클래스 (Lab 3)
- Part 5 · 도구 하나 직접 추가 (Lab 5)

**주어지는 것 (Part 1·6·7)** — 도구 서버, 실행 시나리오, 자동 채점.
마지막 셀 `grade()` 가 네 빈칸이 실제로 동작하는지 채점한다.


## 0 · 설치와 준비
실행 순서대로 위에서 아래로. (실제 LLM을 호출하므로 NVIDIA API 토큰이 필요하다)


In [ ]:
%pip install -q fastmcp openai


In [ ]:
# NVIDIA API로 Qwen(LLM)을 부른다. 토큰은 입력창/환경변수로 (하드코딩 금지).
import os, getpass, json
from openai import OpenAI

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...): ")
llm = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)
LLM_MODEL = "qwen/qwen3-next-80b-a3b-instruct"     # non-thinking
print("LLM 준비:", LLM_MODEL)


## Part 1 · 도구 서버 (주어짐)
에이전트가 쓸 도구 세 가지. 읽기 `search_docs` · 행동 `add_task` · 검증 있는 행동 `set_priority`.


In [ ]:
# 에이전트가 쓸 도구 서버 (주어짐). 읽기(search_docs) + 행동(add_task) + 검증(set_priority).
from fastmcp import FastMCP, Client

mcp = FastMCP("MiniAgentTools")
DOCS = [
    {"id": "vacation", "text": "연차는 사용 3영업일 전에 사내포털에서 신청한다."},
    {"id": "remote",   "text": "재택근무는 주 2회까지 가능하며 전날 18시까지 공유한다."},
]
TASKS = []
LEVELS = {"낮음", "보통", "높음"}

@mcp.tool
def search_docs(query: str, k: int = 2) -> list:
    """질문과 관련된 사내 문서를 찾는다 (읽기)"""
    scored = sorted(DOCS, key=lambda d: -sum(w in d["text"] for w in query.split()))
    return scored[:k]

@mcp.tool
def add_task(title: str) -> dict:
    """할 일을 추가한다 (부작용)"""
    TASKS.append({"id": len(TASKS) + 1, "title": title, "level": "보통"})
    return TASKS[-1]

@mcp.tool
def set_priority(task_id: int, level: str) -> dict:
    """작업 우선순위를 바꾼다. level은 낮음·보통·높음 중 하나 (아니면 친절한 에러)."""
    if level not in LEVELS:
        raise ValueError(f"level은 {sorted(LEVELS)} 중 하나여야 합니다. 받은 값: '{level}'")
    for t in TASKS:
        if t["id"] == task_id:
            t["level"] = level
            return t
    raise ValueError(f"{task_id}번 작업이 없습니다.")

print("도구 서버 준비:", ["search_docs", "add_task", "set_priority"])


## Part 2 · 도구 변환 `to_openai_tools` (빈칸)
모델은 MCP 도구를 그대로 못 읽는다. `name·description·inputSchema` 를 OpenAI function 스펙으로 바꿔줘야 한다. — **실습 Lab 1 참고**


In [ ]:
def to_openai_tools(tools):
    # TODO(실습 Lab 1 참고): MCP 도구(name·description·inputSchema)를 OpenAI function 스펙으로 변환하라.
    #   형식: {"type": "function", "function": {"name": ..., "description": ..., "parameters": ...}}
    return [____ for t in tools]


## Part 3 · 에이전트 루프 `run_agent` (빈칸) · 핵심
판단(LLM)→행동(도구)→관찰(결과)을 **더 부를 게 없을 때까지** 반복한다. `max_steps` 로 무한루프를 막고, 도구가 에러를 내면 죽지 말고 그 에러를 되먹여 스스로 고치게 한다. — **실습 Lab 1·2·4 참고**


In [ ]:
async def run_agent(question, max_steps=5):
    async with Client(mcp) as c:
        tools = to_openai_tools(await c.list_tools())
        messages = [{"role": "system", "content": "필요하면 도구로 처리한 뒤 한국어로 간결히 답하라."},
                    {"role": "user", "content": question}]
        for _ in range(max_steps):                       # ① 반복 한도 = 안전장치 (Lab 2)
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=messages, tools=tools, temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            # TODO ②(Lab 1): 도구를 더 안 부르면 최종 답변으로 종료하라.
            if ____:
                return ____
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  [LLM → 도구] {tc.function.name}({args})")
                # TODO ③④(Lab 1·4): 도구를 실행하되, 에러가 나면 죽지 말고
                #   'out = f"오류: ..."' 로 담아 되먹여라(self-correction).
                try:
                    out = ____
                except Exception as e:
                    out = ____
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(out, ensure_ascii=False, default=str)})
        return "(반복 한도 초과)"


## Part 4 · 메모리 `Agent` (빈칸)
`run_agent` 는 매번 대화가 초기화된다. `messages` 를 객체에 담아 여러 질문에 걸쳐 유지하면, '방금 그 작업' 같은 말을 알아듣는다. — **실습 Lab 3 참고**


In [ ]:
class Agent:
    """대화(messages)를 여러 질문에 걸쳐 유지하는 에이전트 (메모리)."""
    def __init__(self, system):
        # TODO(실습 Lab 3 참고): system 메시지 하나로 self.messages 를 시작하라.
        self.messages = ____

    async def ask(self, question, max_steps=5):
        async with Client(mcp) as c:
            tools = to_openai_tools(await c.list_tools())
            # TODO(Lab 3): 이번 질문을 '기존 대화 위에' 쌓아라 (새 리스트로 시작하지 말 것).
            ____
            for _ in range(max_steps):
                m = llm.chat.completions.create(
                    model=LLM_MODEL, messages=self.messages, tools=tools, temperature=0.2, max_tokens=400,
                    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
                ).choices[0].message
                if not m.tool_calls:
                    self.messages.append({"role": "assistant", "content": m.content})
                    return m.content
                self.messages.append({"role": "assistant", "content": m.content or "",
                                      "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
                for tc in m.tool_calls:
                    args = json.loads(tc.function.arguments)
                    print(f"  [LLM → 도구] {tc.function.name}({args})")
                    try:
                        out = (await c.call_tool(tc.function.name, args)).data
                    except Exception as e:
                        out = f"오류: {str(e).splitlines()[-1]}"
                    self.messages.append({"role": "tool", "tool_call_id": tc.id,
                                          "content": json.dumps(out, ensure_ascii=False, default=str)})
            return "(반복 한도 초과)"


## Part 5 · 도구 하나 직접 추가 (빈칸)
좋은 도구는 이름·설명·반환이 분명하다. 지금까지 등록한 할 일을 읽는 `list_tasks` 를 직접 만들어 서버에 붙여라. — **실습 Lab 5 참고**


In [ ]:
# TODO(실습 Lab 5·도구 설계 참고): 읽기 도구 list_tasks 를 직접 만들어라.
#   - @mcp.tool 데코레이터를 붙이고
#   - 인자는 없고, 반환은 TASKS (지금까지 등록된 할 일 리스트)
#   - 모델이 읽는 한 줄 docstring을 달 것
____

print("list_tasks 를 만들었다면, 아래 자동 채점의 보너스 항목에서 확인됩니다.")


## Part 6 · 시나리오 실행 (주어짐)
완성한 에이전트를 멀티스텝·자기수정·메모리 세 상황으로 돌려본다.


In [ ]:
# 완성한 에이전트를 세 가지로 확인한다. (실제 LLM 호출 — 토큰 필요)
print("① 멀티스텝 — 문서 찾고 할 일 추가")
print(await run_agent("연차 신청 규정을 찾아보고, '연차 신청' 을 할 일로 추가해줘"))
print()
print("② 자기수정 — '긴급'은 허용값이 아님 → 에러를 보고 스스로 유효값으로 고침")
print(await run_agent("'발표 준비' 를 추가하고, 그 작업 우선순위를 '긴급'으로 설정해줘"))
print()
print("③ 메모리 — 앞 대화를 이어받아 '방금 그 작업'을 해결")
a = Agent("너는 업무 비서다. 간결히 답하라.")
print(await a.ask("'디자인 리뷰' 를 할 일로 추가해줘"))
print(await a.ask("방금 그 작업 우선순위를 높음으로 바꿔줘"))


## Part 7 · 자동 채점 (주어짐)
네 빈칸이 실제로 동작하는지 돌려서 확인한다. 100점 만점(보너스 +10).


In [ ]:
# 자동 채점 — 네 빈칸이 '동작'하는지 실제로 돌려서 확인한다.
async def grade():
    total, notes = 0, []

    # (1) 도구 변환 · 20
    try:
        async with Client(mcp) as c:
            spec = to_openai_tools(await c.list_tools())
        assert spec and spec[0]["type"] == "function" and "name" in spec[0]["function"]
        total += 20; notes.append("[20] Part2 도구 변환 to_openai_tools ✓")
    except Exception as e:
        notes.append(f"[ 0] Part2 도구 변환 ✗  ({e})")

    # (2) 루프·멀티스텝 · 30 — 도구를 실제로 부르고 결과를 반영
    try:
        TASKS.clear()
        await run_agent("'주간 회의 준비' 를 할 일로 추가해줘")
        assert any("회의" in (t["title"] or "") for t in TASKS), "add_task 가 실행되지 않음"
        total += 30; notes.append("[30] Part3 루프·멀티스텝 (판단→도구→관찰) ✓")
    except Exception as e:
        notes.append(f"[ 0] Part3 루프·멀티스텝 ✗  ({e})")

    # (3) 자기수정 · 25 — 잘못된 우선순위를 에러 보고 스스로 복구
    try:
        TASKS.clear()
        await run_agent("'마감 정리' 를 추가하고, 그 작업 우선순위를 '긴급'으로 설정해줘")
        assert any(t["level"] == "높음" for t in TASKS), "'긴급' 거부 후 유효값으로 복구 못함"
        total += 25; notes.append("[25] Part3 자기수정 (에러 되먹임→복구) ✓")
    except Exception as e:
        notes.append(f"[ 0] Part3 자기수정 ✗  ({e})")

    # (4) 메모리 · 25 — 앞 대화를 이어받아 '방금 그 작업' 해결
    try:
        TASKS.clear()
        a = Agent("너는 업무 비서다.")
        await a.ask("'회고 작성' 을 할 일로 추가해줘")
        await a.ask("방금 그 작업 우선순위를 높음으로")
        assert any(t["level"] == "높음" for t in TASKS), "이전 대화를 참조하지 못함"
        assert len(a.messages) > 3, "messages 가 유지되지 않음"
        total += 25; notes.append("[25] Part4 메모리 (대화 유지→맥락 참조) ✓")
    except Exception as e:
        notes.append(f"[ 0] Part4 메모리 ✗  ({e})")

    # (보너스) list_tasks 도구 등록 · +10
    try:
        async with Client(mcp) as c:
            names = [t.name for t in await c.list_tools()]
        if "list_tasks" in names:
            total += 10; notes.append("[+10] Part5 도구 추가 list_tasks ✓")
        else:
            notes.append("[ +0] Part5 도구 추가 list_tasks 미등록")
    except Exception as e:
        notes.append(f"[ +0] Part5 도구 추가 ✗  ({e})")

    print("\n".join(notes))
    print("-" * 46)
    print(f"점수: {total} / 100   (보너스 최대 +10)")
    return total

await grade()


## 마무리 · 여기서 더 나아가려면
- **RAG를 도구로** — `search_docs` 를 임베딩 벡터검색으로 바꿔 더 똑똑하게 (실습 Lab 6)
- **코드 에이전트** — 코드를 실행하는 도구를 붙여 스스로 고치는 루프 (실습 Lab 7)
- **모델 스위칭** — 같은 에이전트를 다른 모델로 갈아끼워 비교 (실습 Lab 8)
- **배포** — 도구 서버를 `server.py` 로 떼어내 `Client("http://...")` 로 붙이기
